# Scaling-factor distribution scanner

Standalone diagnostic notebook for picking normalization caps that are
**physically consistent** (fixed, symmetric around zero, with safety
headroom) but **not blind to the data** (informed by the actual operating
envelope of the training trajectories).

This notebook does **not** import from `train_PINN_GPU_6GB.py`. The filename
regex, whitelist parser, Arrow column lists, and current `state_max / force_max
/ control_max` values are reproduced verbatim at the top so the scan sees the
same files the PINN sees. If you change any of those in the trainer, update
the corresponding constants here and re-run.

## What this measures

For each scanned variable, three independent distributions are computed:

- **first** — the first `WINDOW_SECONDS` of each trajectory
- **last** — the last `WINDOW_SECONDS` of each trajectory
- **combined** — the full trajectory

The first/last split exists to surface how the desired-yaw-rate ramp changes
the operating envelope between trajectory halves. Caps must cover the
**combined** window (the trainer sees the whole thing), but the first-vs-last
comparison tells you which variables are sensitive to the yaw schedule and
which aren't.

## What gets a recommendation

For every variable, two suggested caps:

- **safe** = `|max| × 1.30`, snapped to a nice number. The default. Covers the
  full data envelope with a 30% engineering margin and never clips.
- **tight** = `|p99.9| × 1.10`. Clips the worst 0.1% of samples — only useful
  if you've inspected the tail and confirmed it's noise rather than physics.
  For a clean simulation dataset, the tail is usually real, so prefer **safe**.

Both are symmetric (capping `|value|`), so scaled values land in `[-1, 1]`
at the peak. Neither uses mean/std, so once you commit the resulting numbers
the normalization stays fixed and applies identically to any future trajectory.


## 1. Imports


In [ ]:
from __future__ import annotations

import re
import json
import math
from pathlib import Path
from collections import defaultdict
from typing import Dict, List, Any, Optional, Tuple
from tqdm.notebook import tqdm

import numpy as np
import pyarrow.feather as feather
import matplotlib.pyplot as plt

%matplotlib inline


## 2. Configuration

Edit these to point at your data and pick a window size.

- `DATA_DIR` — folder containing `*.arrow` files. Same as the trainer's
  `config['data_dir']`.
- `WHITELIST_PATH` — `pinn_training_whitelist.txt`. Same as the trainer.
  If the file doesn't exist or is `None`, every `*.arrow` in `DATA_DIR` is used.
- `WINDOW_SECONDS` — head/tail window length. 10s is the natural choice for
  20s trajectories (covers each half exactly).
- `SAFETY_MARGIN` — multiplier on `|max|` for the "safe" cap. 1.30 = 30%
  engineering headroom over the largest value seen.
- `ROUND_SUGGESTIONS` — snap suggestions to {1, 1.5, 2, 3, 5, 7} × 10ⁿ so
  recommended caps look like `1.5` instead of `1.473`.


In [ ]:
# Edit these.
DATA_DIR: Path = Path(r"SimulationDataSlipSpin_Julia_3")
WHITELIST_PATH: Optional[Path] = None #Path(r"G:\My Drive\pinn_training_whitelist.txt")

WINDOW_SECONDS: float = 10.0
SAFETY_MARGIN: float = 1.30
ROUND_SUGGESTIONS: bool = True

# Microslip diagnostic: fraction of samples with |Vp| below this threshold.
# This is the friction-band cutoff — when |Vp| < eps, the tanh(Vp/eps)
# friction smoother is nearly linear (no Coulomb plateau). If a large
# fraction of training data sits below this value, your tanh envelope
# is wider than the actual operating slip and friction is being learned
# in the wrong regime. Tighten eps until the below-threshold fraction
# drops to a small percentage of the worst-case trajectory.
MICROSLIP_THRESHOLD: float = 0.01  # m/s

# Subsample target per (file, window) for percentile estimation. Running
# scalars (min, max, count, mean, std) are computed exactly across ALL
# samples; only the percentile estimator uses this subsample. 1000 samples
# per file × ~hundreds of files = >100k samples per (var, window), which
# gives p99.9 estimates accurate to within ~5%.
SUBSAMPLE_PER_FILE: int = 1000

# Output paths (relative to notebook directory).
PLOT_PATH: str = "scaling_distribution_windowed_withtheta.png"
JSON_PATH: str = "scaling_summary_windowed_withtheta.json"


## 3. Trainer-equivalent constants

These are duplicated from `train_PINN_GPU_6GB.py` so the notebook stays
standalone. **If you change any of these in the trainer, update them here
and re-run.**

- `_FNAME_RE` — regex matching the simulator's output filenames
- `_ARROW_*_COLS` — column names the trainer pulls from each Arrow file
- `_STATE_MAX, _FORCE_MAX, _CONTROL_MAX` — current normalization caps used
  for the "current" column in the summary tables


In [ ]:
_FNAME_RE = re.compile(
    r'^test_mu_(?P<mu>[0-9.]+)_(?P<motion>[a-z]+)_asmc_case(?P<fc>\d+)'
    r'_psi_var_beta(?P<beta>[-0-9.]+)_amp(?P<amp>[0-9.]+)_chi_(?P<chi>[0-9.]+)\.arrow$'
)

_ARROW_STATE_COLS   = ['Vx', 'Vy', 'psi_dot',
                       'w1', 'w2', 'w3', 'w4',
                       'theta1', 'theta2', 'theta3', 'theta4']
_ARROW_CONTROL_COLS = ['Msat_1', 'Msat_2', 'Msat_3', 'Msat_4']
_ARROW_FORCE_COLS   = ['Fx_1', 'Fx_2', 'Fx_3', 'Fx_4',
                       'Fy_1', 'Fy_2', 'Fy_3', 'Fy_4',
                       'Mz_1', 'Mz_2', 'Mz_3', 'Mz_4']
_ARROW_GAMMA_COLS   = ['gamma1', 'gamma2', 'gamma3', 'gamma4']  # roller spin angular velocities γ̇_i (rad/s)
_ARROW_VP_COLS      = ['Vpx_1', 'Vpx_2', 'Vpx_3', 'Vpx_4',
                       'Vpy_1', 'Vpy_2', 'Vpy_3', 'Vpy_4']  # per-wheel contact-point slip velocity components (m/s)
_ARROW_TIME_COL     = 'time'

_STATE_MAX = np.array([1.0, 1.0, 2.0,
                       15., 15., 15., 15.,
                       1., 1., 1., 1.], dtype=np.float32)
_F_MAX  = 35.6 * 9.81
_MZ_MAX = _F_MAX / 100 / 4.0
_FORCE_MAX   = np.array([_F_MAX]*8 + [_MZ_MAX]*4, dtype=np.float32)
_CONTROL_MAX = np.array([10., 10., 10., 10.], dtype=np.float32)


## 4. Variables to scan

| group | variables | why included |
|---|---|---|
| platform velocities | `Vx, Vy, psi_dot` | core state — yaw rate is the one that ramps |
| wheel angular velocities | `w1..w4` | derived from V and yaw via kinematics |
| friction forces | `Fx_1..4, Fy_1..4` | dec1 / dec3 supervised target |
| spin friction torque | `Mz_1..4` | the χ-sensitive component |
| control torques | `Msat_1..4` | network input, normalized too |
| wheel rotation angles | `theta_1..4` | included to confirm cumulative range; folded to ±π/12 in data.py before reaching the loss |

| group | variables | reason for skipping |
|---|---|---|
| roller angular velocities | `gamma_1..4` | not in the PINN's 11-D state vector. added here for force sensitivity to fold angle error |
| global position | `x_o, y_o` | not in the PINN's state vector |

The `current_cap` column shows whatever is currently committed in the trainer
(or in the constants above, if you've edited them).


In [ ]:
VARS_TO_SCAN: List[Tuple[str, str, int, str, float]] = [
    # name,    source,    idx, units,   current_cap
    ('Vx',      'state',    0, 'm/s',   _STATE_MAX[0]),
    ('Vy',      'state',    1, 'm/s',   _STATE_MAX[1]),
    ('psi_dot', 'state',    2, 'rad/s', _STATE_MAX[2]),
    ('w1',      'state',    3, 'rad/s', _STATE_MAX[3]),
    ('w2',      'state',    4, 'rad/s', _STATE_MAX[4]),
    ('w3',      'state',    5, 'rad/s', _STATE_MAX[5]),
    ('w4',      'state',    6, 'rad/s', _STATE_MAX[6]),
    ('theta1',  'state',    7, 'rad',   _STATE_MAX[7]),
    ('theta2',  'state',    8, 'rad',   _STATE_MAX[8]),
    ('theta3',  'state',    9, 'rad',   _STATE_MAX[9]),
    ('theta4',  'state',   10, 'rad',   _STATE_MAX[10]),

    ('Fx_1',    'force',    0, 'N',     _FORCE_MAX[0]),
    ('Fx_2',    'force',    1, 'N',     _FORCE_MAX[1]),
    ('Fx_3',    'force',    2, 'N',     _FORCE_MAX[2]),
    ('Fx_4',    'force',    3, 'N',     _FORCE_MAX[3]),
    ('Fy_1',    'force',    4, 'N',     _FORCE_MAX[4]),
    ('Fy_2',    'force',    5, 'N',     _FORCE_MAX[5]),
    ('Fy_3',    'force',    6, 'N',     _FORCE_MAX[6]),
    ('Fy_4',    'force',    7, 'N',     _FORCE_MAX[7]),
    ('Mz_1',    'force',    8, 'N·m',   _FORCE_MAX[8]),
    ('Mz_2',    'force',    9, 'N·m',   _FORCE_MAX[9]),
    ('Mz_3',    'force',   10, 'N·m',   _FORCE_MAX[10]),
    ('Mz_4',    'force',   11, 'N·m',   _FORCE_MAX[11]),

    ('Msat_1',  'control',  0, 'N·m',   _CONTROL_MAX[0]),
    ('Msat_2',  'control',  1, 'N·m',   _CONTROL_MAX[1]),
    ('Msat_3',  'control',  2, 'N·m',   _CONTROL_MAX[2]),
    ('Msat_4',  'control',  3, 'N·m',   _CONTROL_MAX[3]),

    ('gamma1',  'gamma',    0, 'rad/s', float('nan')),
    ('gamma2',  'gamma',    1, 'rad/s', float('nan')),
    ('gamma3',  'gamma',    2, 'rad/s', float('nan')),
    ('gamma4',  'gamma',    3, 'rad/s', float('nan')),

    # Per-wheel slip-velocity magnitudes |Vp_i| = sqrt(Vpx_i^2 + Vpy_i^2).
    # Diagnostic only — no cap is learned from these; they drive the
    # friction-envelope (tanh) eps decision via MICROSLIP_THRESHOLD.
    ('Vp_norm_1', 'vp_norm', 0, 'm/s', float('nan')),
    ('Vp_norm_2', 'vp_norm', 1, 'm/s', float('nan')),
    ('Vp_norm_3', 'vp_norm', 2, 'm/s', float('nan')),
    ('Vp_norm_4', 'vp_norm', 3, 'm/s', float('nan')),
]

WINDOWS = ['first', 'last', 'combined']
WINDOW_COLOURS = {
    'first':    '#1E3A5F',  # navy
    'last':     '#D97706',  # amber
    'combined': '#64748B',  # slate
}

print(f"{len(VARS_TO_SCAN)} variables will be scanned across {len(WINDOWS)} windows.")


## 5. Helper functions

Three small utilities:

- `parse_arrow_filename` extracts μ, motion case, friction case, β, amplitude,
  and χ from the simulator's filename pattern.
- `parse_whitelist` reads `pinn_training_whitelist.txt`. Lines starting with
  `#` are comments. Returns `None` if the file doesn't exist (which means
  "no whitelist; use everything").
- `round_nice` snaps a floating-point cap suggestion to a friendly number
  drawn from {1, 1.5, 2, 3, 5, 7} × 10ⁿ.
- `compute_stats` produces min, max, |max|, mean, std, and several percentiles
  (including |·| percentiles, which are the right ones to look at for a
  symmetric cap).


In [ ]:
def parse_arrow_filename(name: str) -> Optional[Dict[str, Any]]:
    m = _FNAME_RE.match(name)
    if not m:
        return None
    return {
        'mu':     float(m['mu']),
        'motion': m['motion'],
        'fc':     int(m['fc']),
        'beta':   float(m['beta']),
        'amp':    float(m['amp']),
        'chi':    float(m['chi']),
    }


def parse_whitelist(path: Optional[Path]) -> Optional[set]:
    if path is None or not Path(path).exists():
        if path is not None:
            print(f"[whitelist] {path} not found — using all *.arrow in DATA_DIR")
        return None
    keep: List[str] = []
    with open(path) as fh:
        for line in fh:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            keep.append(line)
    print(f"[whitelist] {len(keep)} approved trajectories from {path}")
    return set(keep)


def round_nice(x: float) -> float:
    if x == 0 or not np.isfinite(x):
        return x
    sign = math.copysign(1.0, x)
    mag  = abs(x)
    p10  = math.floor(math.log10(mag))
    base = 10 ** p10
    norm = mag / base
    nice = np.array([1.0, 1.5, 2.0, 3.0, 5.0, 7.0, 10.0])
    snapped = nice[min(np.searchsorted(nice, norm, side='left'), len(nice)-1)]
    return sign * snapped * base


def compute_stats(rec: Dict[str, Any]) -> Dict[str, float]:
    """Stats from a streaming record. Exact scalars come from running
    accumulators; percentiles are estimated from the subsample."""
    if rec is None or rec.get('count', 0) == 0:
        return {k: float('nan') for k in
                ['n', 'min', 'max', 'abs_max', 'mean', 'std',
                 'p1', 'p50', 'p99', 'p99_9', 'abs_p99', 'abs_p99_9']}
    samples = rec['samples']
    if samples.size == 0:
        # Have running scalars but no subsamples — happens for empty windows.
        return {
            'n':       rec['count'],
            'min':     rec['min'],
            'max':     rec['max'],
            'abs_max': rec['abs_max'],
            'mean':    rec['mean'],
            'std':     rec['std'],
            'p1':       float('nan'), 'p50':   float('nan'),
            'p99':      float('nan'), 'p99_9': float('nan'),
            'abs_p99':  float('nan'), 'abs_p99_9': float('nan'),
        }
    abs_samples = np.abs(samples)
    return {
        'n':          rec['count'],
        'min':        rec['min'],
        'max':        rec['max'],
        'abs_max':    rec['abs_max'],
        'mean':       rec['mean'],
        'std':        rec['std'],
        'p1':         float(np.quantile(samples, 0.01)),
        'p50':        float(np.quantile(samples, 0.50)),
        'p99':        float(np.quantile(samples, 0.99)),
        'p99_9':      float(np.quantile(samples, 0.999)),
        'abs_p99':    float(np.quantile(abs_samples, 0.99)),
        'abs_p99_9':  float(np.quantile(abs_samples, 0.999)),
    }


## 6. The windowed loader (streaming, low-memory)

Reads every `*.arrow` matched by the whitelist, slices each trajectory by
its time column, and accumulates statistics for the three windows. The
implementation is deliberately memory-bounded so it scales to datasets that
don't fit in RAM:

- **Running scalars are exact, computed across ALL samples**: per `(variable,
  window)` we accumulate `count`, `sum`, `sum_sq` (in float64 to avoid
  catastrophic cancellation), `min`, `max`, and `abs_max`. Memory per accumulator
  is constant — a handful of floats per variable per window regardless of
  trajectory length.
- **Percentile estimation uses a stride subsample**: per file, per window,
  we keep `~SUBSAMPLE_PER_FILE` samples drawn at a fixed stride. With ~hundreds
  of trajectories the aggregated subsample is well over 100k values per
  `(variable, window)`, which is enough for p99.9 to within a few percent.

This means the absolute peak / extremes (`min`, `max`, `abs_max`) are *exact*,
the central tendency (`mean`, `std`) is *exact*, and only the percentiles
(`p1`, `p50`, `p99`, `p99.9`) are estimated. For setting normalization caps
this is the right trade-off — caps care about extremes, which we measure
exactly.

Slicing rules:

- **first** window — `t ≤ t₀ + WINDOW_SECONDS`
- **last** window — `t ≥ t_end − WINDOW_SECONDS`
- **combined** — every sample

If a trajectory is shorter than `2 × WINDOW_SECONDS`, the first and last
windows overlap. The script counts how many of those there are and prints a
single notification rather than skipping them — the **combined** window's
stats are valid regardless.


In [ ]:
def load_windowed(data_dir: Path,
                  whitelist: Optional[set],
                  window_seconds: float,
                  subsample_per_file: int,
                  ) -> Tuple[Dict[str, Dict[str, Dict[str, Any]]], Dict[str, Any]]:
    """Single-pass streaming loader (vectorized, memory-safe).

    Per (source, window), reductions are batched across all columns in one
    numpy call. Windows are contiguous slices (no boolean masking copies).
    Float64 accumulation happens via the dtype= argument of .sum() rather
    than materialising a float64 array.

    The strided subsample is explicitly .copy()'d so the per-file source
    arrays (S, U, F, G, VPN) can be garbage-collected after each file —
    otherwise the samples lists would silently hold every file's full
    arrays alive via view-references, ballooning memory by 10–30x.
    """
    paths = sorted(Path(data_dir).glob('*.arrow'))
    if not paths:
        raise FileNotFoundError(f"No .arrow files in {data_dir}")

    # Per-(var, window) accumulators.
    acc: Dict[Tuple[str, int, str], Dict[str, Any]] = {
        (src, idx, w): {
            'count':   0,
            'sum':     0.0,
            'sum_sq':  0.0,
            'min':     float('inf'),
            'max':     float('-inf'),
            'abs_max': 0.0,
            'samples': [],
            'count_below': 0,
        }
        for (_n, src, idx, _u, _c) in VARS_TO_SCAN
        for w in WINDOWS
    }

    # Variables grouped by source for batched reductions.
    vars_by_source: Dict[str, List[Tuple[str, int]]] = {}
    for (name, src, idx, *_) in VARS_TO_SCAN:
        vars_by_source.setdefault(src, []).append((name, idx))

    files_kept = 0
    files_skipped = 0
    short_traj_count = 0
    durations: List[float] = []

    for fp in tqdm(paths, desc="Loading trajectories", unit="file"):
        parsed = parse_arrow_filename(fp.name)
        if parsed is None:
            files_skipped += 1
            continue
        if whitelist is not None and fp.name not in whitelist:
            files_skipped += 1
            continue

        try:
            df = feather.read_feather(fp)
            S = df[_ARROW_STATE_COLS  ].to_numpy(dtype=np.float32)
            U = df[_ARROW_CONTROL_COLS].to_numpy(dtype=np.float32)
            F = df[_ARROW_FORCE_COLS  ].to_numpy(dtype=np.float32)
            t = df[_ARROW_TIME_COL    ].to_numpy(dtype=np.float32)
        except (KeyError, ValueError) as e:
            print(f"  [skip] {fp.name}: {e}")
            files_skipped += 1
            continue

        # gamma is diagnostic-only; older Arrow files may not have these columns,
        # so load it separately and leave it None if missing (no whole-file skip).
        try:
            G = df[_ARROW_GAMMA_COLS].to_numpy(dtype=np.float32)
        except (KeyError, ValueError):
            G = None

        # Per-wheel slip-velocity magnitude: |Vp_i| = sqrt(Vpx_i^2 + Vpy_i^2).
        # Same graceful-skip treatment as gamma.
        try:
            VP_xy = df[_ARROW_VP_COLS].to_numpy(dtype=np.float32)   # (N, 8)
            Vpx = VP_xy[:, 0:4]
            Vpy = VP_xy[:, 4:8]
            VPN = np.sqrt(Vpx * Vpx + Vpy * Vpy).astype(np.float32)  # (N, 4)
            del VP_xy, Vpx, Vpy
        except (KeyError, ValueError):
            VPN = None

        # Free the DataFrame ASAP.
        del df

        if t.size < 2:
            files_skipped += 1
            continue

        t0, t_end = float(t[0]), float(t[-1])
        traj_dur  = t_end - t0
        durations.append(traj_dur)
        if traj_dur < 2 * window_seconds:
            short_traj_count += 1

        # Contiguous slice indices — array[slice] is a view, not a copy.
        # Much cheaper than boolean masking, which allocates each time.
        n_first      = int(np.searchsorted(t, t0    + window_seconds, side='right'))
        n_last_start = int(np.searchsorted(t, t_end - window_seconds, side='left'))
        window_slices = {
            'first':    slice(None, n_first),
            'last':     slice(n_last_start, None),
            'combined': slice(None, None),
        }

        sources: Dict[str, np.ndarray] = {'state': S, 'control': U, 'force': F}
        if G   is not None: sources['gamma']   = G
        if VPN is not None: sources['vp_norm'] = VPN

        # One batched pass per (source, window): all columns of that source
        # are reduced in a single numpy call. Collapses 105 inner iterations
        # into 15, and lets numpy's column-axis reductions use SIMD.
        for src_name, arr_src in sources.items():
            var_list = vars_by_source.get(src_name)
            if not var_list:
                continue
            for w_name, sl in window_slices.items():
                arr = arr_src[sl]                       # view of arr_src
                n_w = arr.shape[0]
                if n_w == 0:
                    continue

                # All-column reductions, batched. dtype=np.float64 makes the
                # accumulation happen in float64 *without* materialising a
                # float64 copy of arr.
                col_sum    = arr.sum(axis=0, dtype=np.float64)
                col_sum_sq = np.einsum('ij,ij->j', arr, arr, dtype=np.float64)
                col_min    = arr.min(axis=0)
                col_max    = arr.max(axis=0)
                col_abs    = np.maximum(np.abs(col_min), np.abs(col_max))

                # Stride-subsample once per (source, window). The .copy() is
                # CRITICAL — without it, arr_sub remains a view of the
                # per-file source array (S/U/F/G/VPN) and the samples lists
                # would hold every file's full arrays alive. With ~5k files
                # at 1000 Hz that silently accumulates 30+ GB. The .copy()
                # detaches the subsample so per-file sources are freeable.
                stride  = max(1, n_w // max(1, subsample_per_file))
                arr_sub = arr[::stride].copy()          # owned float32 (n_sub, ncols)

                # Microslip mask: one boolean pass per window for vp_norm,
                # then per-column .sum() is cheap. Skip for other sources.
                below_counts = None
                if src_name == 'vp_norm':
                    below_counts = (arr < MICROSLIP_THRESHOLD).sum(axis=0)

                # Push per-column results into per-variable accumulators.
                # arr_sub[:, idx] is a view INTO THE OWNED COPY — safe to
                # keep, since the copy isn't tied to arr_src anymore.
                for (_name, idx) in var_list:
                    a = acc[(src_name, idx, w_name)]
                    a['count']  += int(n_w)
                    a['sum']    += float(col_sum[idx])
                    a['sum_sq'] += float(col_sum_sq[idx])
                    lo = float(col_min[idx]); hi = float(col_max[idx])
                    if lo < a['min']:     a['min']     = lo
                    if hi > a['max']:     a['max']     = hi
                    am = float(col_abs[idx])
                    if am > a['abs_max']: a['abs_max'] = am
                    a['samples'].append(arr_sub[:, idx])    # view of owned copy
                    if below_counts is not None:
                        a['count_below'] += int(below_counts[idx])

        # Free per-file source arrays. Because the samples lists now hold
        # views of detached copies (not views of these arrays), the GC can
        # actually reclaim them now.
        del S, U, F, t, window_slices, sources
        if G   is not None: del G
        if VPN is not None: del VPN
        files_kept += 1

    if files_kept == 0:
        raise RuntimeError(f"All {len(paths)} files were skipped or unreadable.")

    if short_traj_count > 0:
        print(f"[load] note: {short_traj_count}/{files_kept} trajectories "
              f"are shorter than 2*W={2*window_seconds}s; "
              f"their first/last windows overlap.")

    print(f"[load] kept={files_kept}  skipped={files_skipped}  "
          f"window={window_seconds}s  subsample/file={subsample_per_file}")
    print(f"[load] trajectory durations: "
          f"min={min(durations):.2f}s  median={np.median(durations):.2f}s  "
          f"max={max(durations):.2f}s")

    # Finalise: concat subsamples and compute exact mean/std from running sums.
    values_by_name: Dict[str, Dict[str, Dict[str, Any]]] = {}
    for (name, src, idx, _u, _c) in VARS_TO_SCAN:
        values_by_name[name] = {}
        for w in WINDOWS:
            a = acc[(src, idx, w)]
            n = a['count']
            if n == 0:
                values_by_name[name][w] = {
                    'count': 0, 'min': float('nan'), 'max': float('nan'),
                    'abs_max': float('nan'), 'mean': float('nan'),
                    'std': float('nan'),
                    'samples': np.empty(0, dtype=np.float32),
                    'count_below': 0,
                    'frac_below':  float('nan'),
                }
                continue
            mean = a['sum'] / n
            var  = max(0.0, a['sum_sq'] / n - mean * mean)
            std  = math.sqrt(var)
            samples = (np.concatenate(a['samples']) if a['samples']
                       else np.empty(0, dtype=np.float32))
            values_by_name[name][w] = {
                'count':       n,
                'min':         a['min'],
                'max':         a['max'],
                'abs_max':     a['abs_max'],
                'mean':        mean,
                'std':         std,
                'samples':     samples,
                'count_below': a['count_below'],
                'frac_below':  (a['count_below'] / n) if n > 0 else float('nan'),
            }

    total_subsamples = sum(values_by_name[v[0]][w]['samples'].size
                           for v in VARS_TO_SCAN for w in WINDOWS)
    print(f"[load] subsample memory: ~{total_subsamples * 4 / 1e6:.1f} MB total "
          f"({total_subsamples:,} float32 values across all variables/windows)")

    meta = {
        'files_kept':         files_kept,
        'files_skipped':      files_skipped,
        'short_traj_count':   short_traj_count,
        'window_seconds':     window_seconds,
        'subsample_per_file': subsample_per_file,
        'duration_min':       min(durations),
        'duration_median':    float(np.median(durations)),
        'duration_max':       max(durations),
    }
    return values_by_name, meta

## 7. Run the loader

This is the slow step (it reads every Arrow file in the whitelist). On a
typical 400-trajectory dataset, expect tens of seconds.

After this cell finishes, `values_by_name[name][window]` is a 1-D `np.ndarray`
ready for analysis.


In [ ]:
print(f"[setup] data_dir       = {DATA_DIR}")
print(f"[setup] whitelist_path = {WHITELIST_PATH}")
print(f"[setup] window         = {WINDOW_SECONDS} s")
print(f"[setup] subsample/file = {SUBSAMPLE_PER_FILE}")

whitelist = parse_whitelist(WHITELIST_PATH)
values_by_name, meta = load_windowed(
    data_dir=DATA_DIR,
    whitelist=whitelist,
    window_seconds=WINDOW_SECONDS,
    subsample_per_file=SUBSAMPLE_PER_FILE,
)
print(f"\n[done] {len(values_by_name)} variables loaded across {len(WINDOWS)} windows.")


## 8. Per-window summary tables

Three tables, one per window. Read them top-to-bottom — each row is a
variable. The columns:

| column | meaning |
|---|---|
| `current` | the normalization cap currently committed (from `_STATE_MAX` / `_FORCE_MAX` / `_CONTROL_MAX`) |
| `min`, `max` | extreme values seen in this window |
| `\|max\|` | maximum of `\|value\|` — the symmetric peak |
| `p99.9` | 99.9 percentile of the **signed** value (gives a sense of the upper-tail edge) |
| `mean`, `std` | central tendency |

The `combined` table is what determines the recommended caps further down,
since the trainer sees the whole trajectory. The `first` and `last` tables
exist to expose how the operating envelope changes between the two halves.


In [ ]:
def print_table_per_window(values_by_name, window):
    print('\n' + '=' * 110)
    print(f"WINDOW = {window}")
    print('=' * 110)
    print(f"{'variable':<10s} {'units':<6s} "
          f"{'current':>9s} {'min':>10s} {'max':>10s} {'|max|':>10s} "
          f"{'p99.9':>10s} {'mean':>9s} {'std':>9s} {'frac<eps':>9s}")
    print('-' * 110)
    for (name, src, idx, units, current) in VARS_TO_SCAN:
        rec = values_by_name[name][window]
        s = compute_stats(rec)
        # frac<eps is meaningful only for vp_norm rows. For other rows we
        # print '-' to make it visually obvious the column doesn't apply.
        if src == 'vp_norm':
            frac_str = f"{rec.get('frac_below', float('nan'))*100:>8.2f}%"
        else:
            frac_str = f"{'-':>9s}"
        print(f"{name:<10s} {units:<6s} "
              f"{current:>9.3g} {s['min']:>10.3g} {s['max']:>10.3g} "
              f"{s['abs_max']:>10.3g} {s['abs_p99_9']:>10.3g} "
              f"{s['mean']:>9.2g} {s['std']:>9.2g} {frac_str}")

print_table_per_window(values_by_name, 'first')


In [ ]:
print_table_per_window(values_by_name, 'last')


In [ ]:
print_table_per_window(values_by_name, 'combined')


## 8b. Microslip diagnostic — friction-envelope sizing

For each wheel and each window, the fraction of timesteps where the
contact-point slip velocity magnitude `|Vp_i|` is below
`MICROSLIP_THRESHOLD`. **A large fraction is a red flag** for a
`tanh(|Vp|/eps)`-style friction smoother with `eps ≈ MICROSLIP_THRESHOLD`:
it means the trajectory spends most of its time in the linear part of
the tanh, never hitting the Coulomb plateau — so the model is learning
the wrong friction regime. Tighten `eps` (smaller threshold) until the
below-threshold fraction is a small fraction of the worst-case wheel /
window combination, then verify the trainer's friction loss still has
smooth gradients there.


In [ ]:
def print_microslip_table(values_by_name, threshold):
    vp_vars = [v for v in VARS_TO_SCAN if v[1] == 'vp_norm']
    if not vp_vars:
        print('[microslip] no Vp_norm rows in VARS_TO_SCAN; nothing to do.')
        return
    print('\n' + '=' * 100)
    print(f"MICROSLIP DIAGNOSTIC  —  threshold = {threshold:g} m/s")
    print("(fraction of timesteps with |Vp_i| below threshold; smaller is better")
    print(" for a tanh friction envelope sized near this threshold)")
    print('=' * 100)
    print(f"{'wheel':<10s} {'window':<10s} "
          f"{'n_total':>12s} {'n_below':>12s} {'frac<eps':>10s}  "
          f"{'median|Vp|':>12s} {'p99|Vp|':>10s} {'max|Vp|':>10s}")
    print('-' * 100)
    worst = (0.0, None)        # (frac, (name, window))
    for (name, src, idx, units, _c) in vp_vars:
        for w in WINDOWS:
            rec = values_by_name[name][w]
            n   = rec['count']
            nb_ = rec.get('count_below', 0)
            fr  = rec.get('frac_below', float('nan'))
            s   = compute_stats(rec)
            samples = rec['samples']
            med = float(np.quantile(samples, 0.5)) if samples.size else float('nan')
            p99 = float(np.quantile(samples, 0.99)) if samples.size else float('nan')
            print(f"{name:<10s} {w:<10s} "
                  f"{n:>12,d} {nb_:>12,d} {fr*100:>9.2f}%  "
                  f"{med:>12.4g} {p99:>10.4g} {s['max']:>10.4g}")
            if np.isfinite(fr) and fr > worst[0]:
                worst = (fr, (name, w))
    print('-' * 100)
    if worst[1] is not None:
        wn, ww = worst[1]
        print(f"worst case: {wn} in window '{ww}' — {worst[0]*100:.2f}% of samples "
              f"below |Vp|={threshold:g} m/s")
        if worst[0] > 0.5:
            print("  ↳ ENVELOPE TOO WIDE: majority of slip samples sit in the linear")
            print("    band — tanh is barely Coulomb-like. Tighten eps.")
        elif worst[0] > 0.2:
            print("  ↳ envelope partially active: meaningful fraction in linear band.")
            print("    Consider a smaller eps, or accept and document the regime.")
        else:
            print("  ↳ envelope mostly outside microslip band — eps looks reasonable.")

print_microslip_table(values_by_name, MICROSLIP_THRESHOLD)


## 9. First-vs-last delta

Probably the most diagnostic table in the notebook. For each variable it
shows the ratio of `|max|` and `std` between the last and first windows.

Interpretation:

- **ratio ≈ 1.0** → the variable's envelope is the same in both halves; the
  yaw-rate ramp doesn't move it.
- **ratio > 2 or ratio < 0.5** → the envelope changes substantially between
  halves. Flagged with `←—`. These are the variables whose cap must be
  driven by the larger half, not just the average behaviour.
- **`std` ratio without `|max|` ratio change** → the spread changes but the
  peak doesn't. Means the variable spends *more time near its peak* in one
  half than the other.

Expected patterns:

- `psi_dot` and the asymmetric force components (often `Fy`) shift the most,
  since the desired yaw is what's changing.
- `Vx, Vy` may keep the same `|max|` but change *sign distribution* between
  halves — the magnitude bound stays put, the centroid moves.
- Wheel speeds `w_i` may cluster bimodally if the yaw flips wheel direction
  partway through.


In [ ]:
def print_delta_table(values_by_name):
    print('\n' + '=' * 95)
    print("FIRST vs LAST window — operating envelope shift (ratio = last / first)")
    print('=' * 95)
    print(f"{'variable':<10s} {'units':<6s} "
          f"{'|max|_first':>12s} {'|max|_last':>12s} {'|max|_ratio':>12s}   "
          f"{'std_first':>10s} {'std_last':>10s} {'std_ratio':>10s}")
    print('-' * 95)
    for (name, src, idx, units, _c) in VARS_TO_SCAN:
        sf = compute_stats(values_by_name[name]['first'])
        sl = compute_stats(values_by_name[name]['last'])
        m_ratio = (sl['abs_max'] / sf['abs_max']) if sf['abs_max'] > 0 else float('nan')
        s_ratio = (sl['std']     / sf['std'])     if sf['std']     > 0 else float('nan')
        flag = ''
        if np.isfinite(m_ratio) and (m_ratio > 2.0 or m_ratio < 0.5):
            flag = '  ←—'
        print(f"{name:<10s} {units:<6s} "
              f"{sf['abs_max']:>12.3g} {sl['abs_max']:>12.3g} {m_ratio:>12.2f}   "
              f"{sf['std']:>10.2g} {sl['std']:>10.2g} {s_ratio:>10.2f}{flag}")

print_delta_table(values_by_name)


## 10. Cap recommendations

Based on the **combined** window stats, since that's what the trainer sees.

For each variable:

- **safe** = `|max| × 1.30`, snapped to nice number — covers the full envelope
  with 30% headroom. This is the default to use.
- **tight** = `|p99.9| × 1.10`, snapped — clips the worst 0.1% of samples.
  Only meaningful if the tail is genuine outlier/noise, which for a clean
  simulation dataset is usually not the case. Use `safe` unless you've
  inspected the histograms below.

The `note` column tells you whether the current cap is healthy or not:

- *"current cap is N× larger than peak — too loose"* → you're diluting the
  gradient signal on this variable. Tighten it.
- *"data exceeds current cap"* → values are being clipped during training.
  Loosen.
- *"current cap utilisation = N%"* → reasonable; the cap covers the data with
  some headroom.


In [ ]:
def suggest_cap(stats_combined, current):
    safe  = stats_combined['abs_max']    * SAFETY_MARGIN
    tight = stats_combined['abs_p99_9']  * 1.10
    if ROUND_SUGGESTIONS:
        safe  = round_nice(safe)
        tight = round_nice(tight)

    used = stats_combined['abs_max'] / current if current > 0 else float('inf')
    if used < 0.30:
        note = f"current cap is {1/used:.1f}x larger than peak - too loose"
    elif used > 1.0:
        note = f"data exceeds current cap (peak/cap = {used:.2f}) - too tight!"
    else:
        note = f"current cap utilisation = {used*100:.0f}%"
    return safe, tight, note


def print_recommendations(values_by_name):
    print('\n' + '=' * 95)
    print("RECOMMENDATIONS — based on the COMBINED window")
    print('=' * 95)
    print(f"{'variable':<10s} {'current':>9s} {'safe':>9s} {'tight':>9s}   note")
    print('-' * 95)
    rows = []
    for (name, src, idx, units, current) in VARS_TO_SCAN:
        s_comb = compute_stats(values_by_name[name]['combined'])
        safe, tight, note = suggest_cap(s_comb, current)
        print(f"{name:<10s} {current:>9.3g} {safe:>9.3g} {tight:>9.3g}   {note}")
        rows.append({
            'name': name, 'source': src, 'idx': idx, 'units': units,
            'current_cap': float(current),
            'suggested_safe_cap':  float(safe),
            'suggested_tight_cap': float(tight),
            'note': note,
            'stats': {w: compute_stats(values_by_name[name][w]) for w in WINDOWS},
            # Microslip diagnostic: count and fraction of samples with
            # value < MICROSLIP_THRESHOLD. Only meaningful for 'vp_norm'
            # rows; for other rows the trainer ignores these fields.
            'microslip': {
                'threshold': float(MICROSLIP_THRESHOLD),
                'per_window': {
                    w: {
                        'count_total': int(values_by_name[name][w]['count']),
                        'count_below': int(values_by_name[name][w].get('count_below', 0)),
                        'frac_below':  float(values_by_name[name][w].get('frac_below', float('nan'))),
                    } for w in WINDOWS
                },
            },
        })
    return rows

rows = print_recommendations(values_by_name)


## 11. Distribution plot

One panel per variable, three overlaid density-normalized histograms (so
different sample counts don't bias the visual comparison):

| colour | window |
|---|---|
| navy (filled) | first 10 s |
| amber (filled) | last 10 s |
| slate (light) | combined background |

Red dashed lines mark `± current cap`.

Reading the plot:

- If navy and amber sit on top of each other → the variable's distribution
  doesn't shift between halves.
- If they're laterally shifted → the centroid changes (typically because
  the variable's sign flips with yaw direction).
- If one is wider than the other → the spread changes between halves.
- Red lines well outside the histogram → cap is too loose.
- Red lines cutting through the histogram body → cap is too tight; data
  is being clipped during training.


In [ ]:
def plot_distributions(values_by_name, output_path):
    n_vars = len(VARS_TO_SCAN)
    n_cols = 4
    n_rows = (n_vars + n_cols - 1) // n_cols
    fig, axs = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 2.7 * n_rows))
    axs = axs.flatten()

    for ax, (name, src, idx, units, current) in zip(axs, VARS_TO_SCAN):
        # Use subsamples for histograms (memory-bounded), exact stats for
        # title annotations.
        rec_comb = values_by_name[name]['combined']
        all_samples = np.concatenate(
            [values_by_name[name][w]['samples'] for w in WINDOWS])
        finite = all_samples[np.isfinite(all_samples)]
        if finite.size == 0:
            continue
        lo, hi = np.quantile(finite, [0.0005, 0.9995])
        # Make sure the exact extremes are visible too — the subsample may not
        # have caught them.
        lo = min(lo, rec_comb['min'])
        hi = max(hi, rec_comb['max'])
        pad = max(abs(lo), abs(hi)) * 0.05 + 1e-9
        bins = np.linspace(lo - pad, hi + pad, 80)

        for w in WINDOWS:
            v = values_by_name[name][w]['samples']
            v = v[(v >= bins[0]) & (v <= bins[-1])]
            ax.hist(v, bins=bins, color=WINDOW_COLOURS[w],
                    alpha=0.45 if w != 'combined' else 0.20,
                    histtype='stepfilled', density=True,
                    label=f"{w}  (n_sub={v.size:,})")

        ax.axvline( current, color='#B91C1C', linestyle='--', linewidth=1.0)
        ax.axvline(-current, color='#B91C1C', linestyle='--', linewidth=1.0)
        # Mark exact peaks too — these are exact across all samples.
        ax.axvline( rec_comb['abs_max'], color='#0F766E',
                   linestyle=':', linewidth=0.7, alpha=0.7)
        ax.axvline(-rec_comb['abs_max'], color='#0F766E',
                   linestyle=':', linewidth=0.7, alpha=0.7)
        ax.axvline(0, color='#000000', linewidth=0.4, alpha=0.4)

        ax.set_title(f"{name}  [{units}]   "
                     f"|max|={rec_comb['abs_max']:.3g}   cap={current:.3g}",
                     fontsize=9)
        ax.tick_params(labelsize=7)
        ax.legend(fontsize=6, loc='upper right', framealpha=0.85)

    for ax in axs[n_vars:]:
        ax.set_visible(False)

    fig.suptitle(f"Distributions: first {WINDOW_SECONDS:g}s vs last "
                 f"{WINDOW_SECONDS:g}s vs combined  "
                 f"(red dashes = current ±cap, teal dots = exact ±|max|)",
                 fontsize=12, y=1.005)
    plt.tight_layout()
    plt.savefig(output_path, dpi=120, bbox_inches='tight')
    plt.show()
    print(f"\n[plot] saved {output_path}")

plot_distributions(values_by_name, output_path=PLOT_PATH)


## 12. Save the summary to JSON

So you can reload and re-plot later without rerunning the loader.


In [ ]:
out = {
    'config': {
        'data_dir':      str(DATA_DIR),
        'whitelist':     str(WHITELIST_PATH) if WHITELIST_PATH else None,
        'window_seconds': WINDOW_SECONDS,
        'safety_margin': SAFETY_MARGIN,
        'rounded':       ROUND_SUGGESTIONS,
        'microslip_threshold': MICROSLIP_THRESHOLD,
    },
    'load_meta': meta,
    'rows':      rows,
}
with open(JSON_PATH, 'w') as f:
    json.dump(out, f, indent=2, default=float)
print(f"[json] saved {JSON_PATH}")


## 13. Paste-ready snippet

Last cell prints exactly the three `np.array(...)` lines you'd drop in at the
top of `train_PINN_GPU_6GB.py` to replace the current `state_max`,
`force_max`, `control_max`. State indices 7–10 (the `theta_i` wheel angles)
are kept at `1.0` because they're embedded as sin/cos before entering the
network — the raw cap value is irrelevant.

If you'd rather use the **tight** caps, replace `suggested_safe_cap` with
`suggested_tight_cap` in the cell below.


In [ ]:
by_src: Dict[str, List[Tuple[int, float]]] = {'state': [], 'force': [], 'control': []}
for r in rows:
    if r['source'] not in by_src:
        continue                # skip diagnostic-only rows (gamma, vp_norm)
    by_src[r['source']].append((r['idx'], r['suggested_safe_cap']))
for v in by_src.values():
    v.sort()

state_full = [None] * 11
for idx, val in by_src['state']:
    state_full[idx] = val
for i in range(7, 11):  # theta1..theta4 — keep at 1.0 (sin/cos embed)
    state_full[i] = 1.0

print("# ---- copy-paste into train_PINN_GPU_6GB.py ----")
print("state_max = np.array([" +
      ", ".join(f"{v:>5.3g}" for v in state_full) +
      "], dtype=np.float32)")
print("force_max = np.array([" +
      ", ".join(f"{v:>5.3g}" for (_, v) in by_src['force']) +
      "], dtype=np.float32)")
print("control_max = np.array([" +
      ", ".join(f"{v:>5.3g}" for (_, v) in by_src['control']) +
      "], dtype=np.float32)")


## 14. What to do with the output

Once you have the recommendations:

1. **Don't accept them blindly.** Look at the histograms. If a recommended
   cap sits at the very edge of the data with zero margin, you're in clipping
   territory; if it's miles to the right of the histogram's body, the safety
   factor inflated something it shouldn't have (e.g., one stray spike).

2. **Tighten in one go, not piecewise.** The Adam optimizer's per-parameter
   step normalization means absolute loss scale matters less than the
   *relative* balance between loss components. If you tighten only `force_max`
   without touching `state_max`, you're shifting the state-vs-force balance,
   not just amplifying forces. Either commit all the recommended changes or
   keep the current scaling — don't half-do it.

3. **Re-run after data-set changes.** If you add new motion cases or μ/χ
   values to the training set, the operating envelope shifts and the caps
   may need to grow. This notebook should be re-run as a check whenever the
   whitelist changes.

4. **Check the first-vs-last delta carefully** for `psi_dot` and the `Mz_i`
   components. These are the variables most directly tied to the yaw schedule
   and the χ contact-spot radius respectively. Big shifts in their
   distributions between halves are the signal that the model needs to learn
   regime-dependent behaviour, not just a single static mapping.
